# 1. Importing Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.2f}'.format


In [ ]:
from pathlib import Path
from zipfile import ZipFile

archive_path = Path("kaggle_survey_2017_2021.zip")
with ZipFile(archive_path) as archive:
    csv_name = next(name for name in archive.namelist() if name.lower().endswith('.csv'))
    with archive.open(csv_name) as survey_file:
        df = pd.read_csv(survey_file, low_memory=False)

# Remove the embedded question-label row and retain the five survey years.
valid_years = {'2017', '2018', '2019', '2020', '2021'}
df = df[df['-'].astype(str).str.strip().isin(valid_years)].copy()

# Normalize common text-encoding artifacts in categorical columns.
for column in df.select_dtypes(include='object').columns:
    df[column] = df[column].str.replace('â€™', "'", regex=False)


In [ ]:
df.shape


In [ ]:
df.nunique()

# 2. Initial Data Exploration (EDA)

In [ ]:
df.sample(5)
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()
df.shape

In [ ]:
df.isna().sum()
df.duplicated().sum()

# 3. Data Cleaning

## Question-row handling

The source file contains one embedded question-label row. It was already excluded
in the loading step by retaining rows whose year is 2017 through 2021.


In [ ]:
df.shape


All 106,301 respondent rows are retained; no respondent row is dropped here.


In [ ]:
df.duplicated().sum()

In [ ]:
# Identical answer profiles are retained because the merged file has no respondent ID.
# Matching responses are not sufficient evidence that two rows represent the same person.
df.shape


In [ ]:
df.nunique()

In [ ]:
df.shape

In [ ]:
df.shape

In [ ]:
df.nunique()

## Rename Columns

In [ ]:
df.head()

In [ ]:
df = df.rename(columns={
    '-': 'Year',
    'Q1': 'Age_Group',
    'Q2': "Gender",
    "Q3": 'Country',
    'Q4': 'Education',
    'Q5': 'Job_Title',
    'Q6': 'Programming_Experience',
    'Q8': 'Recommended_Language',
    'Q13': 'TPU_Usage_Frequency',
    'Q14_Part_1': 'Data_Visualization_Tools',
    'Q15': 'ML_Experience_Years',
    'Q23': 'Employer_ML_Adoption',
    'Q25': 'Annual_Salary_USD'
})

In [ ]:
df.head()

## Programming Languages

In [ ]:

df = df.rename(columns={
    "Q7_Part_1": "Python",
    "Q7_Part_2": "R",
    "Q7_Part_3": "SQL",
    "Q7_Part_4": "C",
    "Q7_Part_5": "C++",
    "Q7_Part_6": "Java",
    "Q7_Part_7": "Javascript",
    "Q7_Part_8": "Julia",
    "Q7_Part_9": "Swift",
    "Q7_Part_10": "Bash",
    "Q7_Part_11": "MATLAB",
})


language_cols = ["Python","R","SQL","C","C++","Java","Javascript","Julia","Swift","Bash","MATLAB"]


df_languages = df[language_cols].copy()


language_counts = df_languages.stack().value_counts()
print("Language counts:")
print(language_counts)


## Year

In [ ]:
df['Year'].nunique()

In [ ]:
df['Year'] = df['Year'].astype(str).str.strip()

df['Year'] = df['Year'].replace({'2021 ': '2021'})   
df['Year'] = df['Year'].astype(int)

In [ ]:
df['Year'].value_counts()

## Gender

In [ ]:
gender_mapping = {
    'Male': 'Male',
    'Man': 'Male',
    'Female': 'Female',
    'Woman': 'Female'
   
}


df['Gender'] = df['Gender'].map(gender_mapping)

df['Gender'] = df['Gender'].fillna('Other')

df['Gender'].value_counts()

## Country

In [ ]:
# If you want to print all unique countries:
print("\n--- Country ---")
print(df['Country'].dropna().unique())

In [ ]:
country_mapping = {
    'United States Of America': 'United States',
    'United States of America': 'United States',
    'United States': 'United States',
    'United Kingdom Of Great Britain And Northern Ireland': 'United Kingdom',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'United Kingdom': 'United Kingdom',
    'UK': 'United Kingdom',
    'Iran, Islamic Republic Of...': 'Iran',
    'Iran, Islamic Republic of...': 'Iran',
    'Iran': 'Iran',
    'I do not wish to disclose my location' : 'I Prefer Not To Say',
    'I do not wish to disclose my location': 'I Prefer Not To Say',
    'Other': 'Other',
    'Russia': 'Russia',
    'Viet Nam': 'Vietnam',
    "People \'S Republic Of China": 'China',
    "People 's Republic of China": 'China',
    'Republic of China': 'China',
  
    'China': 'China',
    'Hong Kong (S.A.R.)':'Hong Kong',
    'Republic of Korea' : 'South Korea',
    'Republic of Korea': 'South Korea'
}


df['Country'] = df['Country'].replace(country_mapping)

In [ ]:
print("\n--- Country ---")
print(df['Country'].dropna().unique())

In [ ]:
arab_countries = [
    'Egypt', 'Saudi Arabia', 'Algeria', 'Tunisia', 'Morocco', 'Iraq', 'United Arab Emirates'
]

asia_non_arab = [
    'India', 'Indonesia', 'Pakistan', 'Japan', 'China', 'Vietnam', 'Bangladesh',
    'Singapore', 'South Korea', 'Philippines', 'Sri Lanka', 'Malaysia', 'Thailand',
    'Nepal', 'Kazakhstan', 'Taiwan', 'Hong Kong'
]

europe = [
    'Russia', 'Turkey', 'Greece', 'Belgium', 'Poland', 'Italy', 'France', 'Switzerland',
    'Netherlands', 'Ukraine', 'Romania', 'Sweden', 'Denmark', 'Germany', 'Austria',
    'Czech Republic', 'Hungary', 'Finland', 'Norway', 'Ireland', 'Portugal', 'Belarus'
]

americas = [
    'Mexico', 'Brazil', 'Peru', 'Argentina', 'Chile', 'Canada', 'United States'
]

africa_non_arab = [
    'Nigeria', 'South Africa', 'Kenya', 'Uganda', 'Ghana', 'Ethiopia'
]

oceania = [
    'Australia', 'New Zealand'
]


def get_region(country):
    if country in arab_countries:
        return 'Arab Countries'
    elif country in asia_non_arab:
        return 'Asia (Non-Arab)'
    elif country in europe:
        return 'Europe'
    elif country in americas:
        return 'Americas'
    elif country in africa_non_arab:
        return 'Africa (Non-Arab)'
    elif country in oceania:
        return 'Oceania'
    return 'Other / Unclassified'


df['Region'] = df['Country'].apply(get_region)


## Salary

In [ ]:
df['Annual_Salary_USD'].unique()

In [ ]:
import re

def salary_band_midpoint(value):
    """Return a midpoint for closed USD bands; exclude open-ended/non-disclosed bands."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    lowered = text.lower()
    if 'not wish' in lowered or text.startswith('>') or text.endswith('+'):
        return np.nan

    cleaned = text.replace('$', '').replace(',', '').replace(' ', '')
    match = re.fullmatch(r'(\d+)-(\d+)', cleaned)
    if not match:
        return np.nan

    lower, upper = map(float, match.groups())
    # Some historical bands abbreviate the lower value: 10-20,000 or 125-150,000.
    if lower < 1000 <= upper:
        lower *= 1000
    return (lower + upper) / 2


In [ ]:
df['Salary_Midpoint'] = df['Annual_Salary_USD'].apply(salary_band_midpoint)


In [ ]:
df.head()

## Job_Title

In [ ]:
df['Job_Title'].unique()

In [ ]:
job_title_map = {

  
    "Software Developer/Software Engineer": "Software Engineer",
    "Programmer": "Software Engineer",


    
    "Machine Learning Engineer": "Machine Learning Engineer",
    "Statistician": "Statistician",
    "Predictive Modeler": "Data Scientist",
    "nan": "Other",

   
    "Program/Project Manager": "Project Manager",
    "Product/Project Manager": "Project Manager",
    "Product Manager": "Product Manager",
    "Business Analyst": "Business Analyst",
    "Marketing Analyst": "Business Analyst",

    
    
    "Scientist/Researcher": "Research Scientist",
    "Researcher": "Research Scientist",
    
    
    "Developer Relations/Advocacy": "Developer Advocate",
    "Developer Advocate": "Developer Advocate",

 
    "Currently not employed": "Unemployed",
    "Not employed": "Unemployed",

}
df["Job_Title"] = df["Job_Title"].replace(job_title_map)

In [ ]:
df['Job_Title'].unique()

## Programming_Experience

In [ ]:
experience_mapping = {
    # --- 0 years experience ---
    'I Have Never Written Code': 'No Experience',
    'I have never written code': 'No Experience',
    'I Have Never Written Code But I Want To Learn': 'No Experience',
    'I have never written code but I want to learn': 'No Experience',
    'I Have Never Written Code And I Do Not Want To Learn': 'No Experience',
    'I have never written code and I do not want to learn': 'No Experience',
    "I Don'T Write Code To Analyze Data": 'No Experience',
    "I don't write code to analyze data": 'No Experience',

    # --- Less than a year ---
    '< 1 Years': '0-1 Years',
    '< 1 years': '0-1 Years',
    '< 1 Year': '0-1 Years',
    '< 1 year': '0-1 Years',
    'Less Than A Year': '0-1 Years',
    'Less than a year': '0-1 Years',

    # --- 1-2 / 1-3 years ---
    '1-2 Years': '1-3 Years',
    '1-2 years': '1-3 Years',
    '1 To 2 Years': '1-3 Years',
    '1 to 2 years': '1-3 Years',
    '1-3 Years': '1-3 Years',
    '1-3 years': '1-3 Years',

    # --- 3-5 years ---
    '3-5 Years': '3-5 Years',
    '3-5 years': '3-5 Years',
    '3 To 5 Years': '3-5 Years',
    '3 to 5 years': '3-5 Years',

    # --- 5-10 years ---
    '5-10 Years': '5-10 Years',
    '5-10 years': '5-10 Years',
    '6 To 10 Years': '5-10 Years',
    '6 to 10 years': '5-10 Years',

    # --- 10+ years ---
    '10-20 Years': '10+ Years',
    '10-20 years': '10+ Years',
    '20+ Years': '10+ Years',
    '20+ years': '10+ Years',
    '20-30 Years': '10+ Years',
    '20-30 years': '10+ Years',
    '30-40 Years': '10+ Years',
    '30-40 years': '10+ Years',
    '40+ Years': '10+ Years',
    '40+ years': '10+ Years',
    'More Than 10 Years': '10+ Years',
    'More than 10 years': '10+ Years'
}

df['Programming_Experience'] = df['Programming_Experience'].replace(experience_mapping)

In [ ]:
df['Programming_Experience'].unique()

## Education

In [ ]:
education_mapping = {
     "Professional Doctorate": "Doctoral Degree",
   'Professional doctorate': "Doctoral Degree",
   'Doctoral degree': "Doctoral Degree",
    "Some College/University Study Without Earning A Bachelor'S Degree": "Some College",
    "Some college/university study without earning a bachelor's degree":"Some College",
    "No Formal Education Past High School": "High School or Below",
    'No formal education past high school' : "High School or Below",
    "I Did Not Complete Any Formal Education Past High School": "High School or Below",
    'I did not complete any formal education past high school': "High School or Below",
    'nan' : 'I prefer not to answer'
}
df['Education'] = df['Education'].replace(education_mapping)
df['Education'].unique()

## Age

In [ ]:
age_mapping = {
    '18-21': '18-24', '22-24': '18-24',
    '25-29': '25-34', '30-34': '25-34',
    '35-39': '35-44', '40-44': '35-44',
    '45-49': '45-54', '50-54': '45-54',
    '55-59': '55+', '60-69': '55+', '70+': '55+'
}
df['Age_Group'] = df['Age_Group'].replace(age_mapping)


In [ ]:
df['Age_Group'].unique()

In [ ]:
df['Age_Group'].isna().sum()

## Employer_ML_Adoption

In [ ]:
df['Employer_ML_Adoption'].unique()

In [ ]:
ml_adoption_mapping = {
   
    'No (We Do Not Use Ml Methods)': 'No ML Usage',
    'No (we do not use ML methods)': 'No ML Usage',
   
    'We Use Ml Methods For Generating Insights (But Do Not Put Working Models Into Production)': 'ML for Insights Only',
    'We use ML methods for generating insights (but do not put working models into production)': 'ML for Insights Only',

    'We Are Exploring Ml Methods (And May One Day Put A Model Into Production)': 'Exploring ML',
    'We are exploring ML methods (and may one day put a model into production)': 'Exploring ML',
    
    'We Recently Started Using Ml Methods (I.E., Models In Production For Less Than 2 Years)': 'Recently Adopted ML',
    'We recently started using ML methods (i.e., models in production for less than 2 years)': 'Recently Adopted ML',
   
    'We Have Well Established Ml Methods (I.E., Models In Production For More Than 2 Years)': 'Established ML',
     'We have well established ML methods (i.e., models in production for more than 2 years)': 'Established ML',

    'I Do Not Know': 'Unknown',
    'nan' : 'Unknown'
}


df['Employer_ML_Adoption'] = df['Employer_ML_Adoption'].replace(ml_adoption_mapping)

In [ ]:
df['Employer_ML_Adoption'].unique()

# 5. Data Visualization & 6. Insights & Recommendations

In [ ]:
df.sample(5)

## Age

In [ ]:
df['Age_Group'].value_counts()


In [ ]:
df_filtered_age = df[df['Age_Group'].notna()].copy()


In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=df_filtered_age, x='Age_Group', order=df_filtered_age['Age_Group'].value_counts().index, palette='viridis')
plt.title('Distribution of Age Groups') 
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show();

In [ ]:
df_filtered_age.groupby('Gender')['Age_Group'].value_counts()


In [ ]:
df_filtered_age['Age_Group'] = df_filtered_age['Age_Group'].astype(str)
age_gender_pct = df_filtered_age.groupby('Gender')['Age_Group'].value_counts(normalize=True) * 100

age_gender_df = age_gender_pct.reset_index(name='Percentage')

plt.figure(figsize=(12, 8))
ax = sns.barplot(data=age_gender_df, x='Gender', y='Percentage', hue='Age_Group', palette='Set2')
            
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', fontsize=10,  )

plt.title('Age Group Distribution by Gender')
plt.xlabel('Gender')
plt.ylabel('Percentage within gender (%)')
plt.legend(title='Age Group', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show();

## Gender

In [ ]:
df_filtered = df[df['Gender'] != 'Other'].copy()
df_filtered['Gender'].value_counts()


In [ ]:
plt.figure(figsize=(10, 6))
plt.pie(
    df_filtered['Gender'].value_counts(),
    labels=df_filtered['Gender'].value_counts().index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('muted'),
)
plt.title('Gender Distribution Among Binary Responses')
plt.show()


## Job_Title

In [ ]:
df['Job_Title'].value_counts().head(10)

In [ ]:
df_filtered_job = df[(df['Job_Title'] != 'Other') & (df['Job_Title'] != 'Student') & (df['Job_Title'] != 'Unemployed')]
Top_10_Jobs = df_filtered_job['Job_Title'].value_counts().head(10)
plt.figure(figsize=(12,6))
sns.barplot(x=Top_10_Jobs.index, y=Top_10_Jobs.values, palette='coolwarm')
plt.title('Top 10 Job Titles')  
plt.xlabel('Job Title')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show();


In [ ]:
#df_filtered_3 = df[(df['Job_Title'] != 'Other') & (df['Job_Title'] != 'Student')& (df['Job_Title'] != 'Unemployed')]

top_5= df_filtered_job['Job_Title'].value_counts().head(5)


In [ ]:
top_5_jobs = top_5.index.tolist()
df_top_5 = df[df['Job_Title'].isin(top_5_jobs)]

In [ ]:
gender_job_distribution = df_top_5.groupby(['Job_Title', 'Gender']).size().unstack(fill_value=0)

gender_job_long = gender_job_distribution.reset_index().melt(id_vars='Job_Title', var_name='Gender', value_name='Count')

plt.figure(figsize=(12, 6))
sns.barplot(data=gender_job_long, x='Job_Title', y='Count', hue='Gender', palette='Set2')

plt.title('Distribution of Top 5 Job Titles by Gender')
plt.xlabel('Job Title')
plt.ylabel('Number of Respondents')
plt.xticks(rotation=45)
plt.legend(title="Gender")
plt.tight_layout()
plt.show();

In [ ]:
df_filtered_GJ = df[
    (~df['Job_Title'].isin(['Other', 'Student', 'Unemployed']))
    & df['Age_Group'].notna()
].copy()
top_5 = df_filtered_GJ['Job_Title'].value_counts().head(5)
df_top_jobs = df_filtered_GJ[df_filtered_GJ['Job_Title'].isin(top_5.index)]

plt.figure(figsize=(12, 8))
sns.countplot(
    data=df_top_jobs,
    x='Job_Title',
    order=top_5.index,
    hue='Age_Group',
    palette='Set2',
)
plt.title('Age Group Distribution by Top 5 Job Titles')
plt.xlabel('Job Title')
plt.ylabel('Number of Respondents')
plt.xticks(rotation=45)
plt.legend(title='Age Group', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


## Country

In [ ]:
df['Country'].value_counts().head(15)

In [ ]:
country_scope = df[~df['Country'].isin(['I Prefer Not To Say', 'Other'])].copy()
top_countries = country_scope['Country'].value_counts().head(15).index
plt.figure(figsize=(10, 8))
sns.countplot(data=country_scope, y='Country', order=top_countries, palette='flare')
plt.title('Top 15 Countries by Number of Respondents', fontsize=16)
plt.xlabel('Number of Respondents')
plt.ylabel('Country')
plt.tight_layout()
plt.show()


In [ ]:
df['Region'].value_counts()

In [ ]:
region_order = df['Region'].value_counts().index

plt.figure(figsize=(10, 6))
sns.countplot(data=df, y='Region', order=region_order, palette='Blues_r')

plt.title("Number of Respondents by Region", fontsize=16, fontweight='bold')
plt.xlabel("Number of Respondents")
plt.ylabel("Region")
plt.tight_layout()
plt.show();

In [ ]:
arab_df = df[df['Country'].isin(arab_countries)]
arab_df['Country'].value_counts()
plt.figure(figsize=(10, 6))
sns.countplot(data=arab_df, y='Country', order=arab_df['Country'].value_counts().index, palette='viridis')
plt.title("Number of Respondents in Arab Countries", fontsize=16, fontweight='bold')
plt.xlabel("Number of Respondents")
plt.ylabel("Country")
plt.tight_layout()
plt.show();

In [ ]:
Asia_df = df[df['Country'].isin(asia_non_arab)]
Asia_df['Country'].value_counts()
plt.figure(figsize=(10, 6))
sns.countplot(data=Asia_df, y='Country', order=Asia_df['Country'].value_counts().index, palette='viridis')
plt.title("Number of Respondents in Asia (Non-Arab Countries)", fontsize=16, fontweight='bold')
plt.xlabel("Number of Respondents")
plt.ylabel("Country")
plt.tight_layout()
plt.show();

## Salary_Midpoint

In [ ]:
df.head()

In [ ]:
# Average salaries in the world
global_avg = df['Salary_Midpoint'].mean()

# Average salaries in Arab countries
arab_countries = ['Egypt', 'Saudi Arabia', 'Algeria', 'Tunisia',
                  'Morocco', 'Iraq', 'United Arab Emirates']
arab_avg = df[df['Country'].isin(arab_countries)]['Salary_Midpoint'].mean()

print(f"Average salaries in the world: ${global_avg:,.0f}")
print(f"Average salaries in Arab countries: ${arab_avg:,.0f}")

In [ ]:
salary_by_gender = df[df['Gender'].isin(['Female', 'Male'])].groupby('Gender')['Salary_Midpoint'].mean().dropna()
plt.figure(figsize=(8, 5))
sns.barplot(x=salary_by_gender.index, y=salary_by_gender.values, palette='Set2')
plt.title('Estimated Salary Midpoint by Gender')
plt.xlabel('Gender')
plt.ylabel('Estimated salary midpoint (USD)')
plt.show()
# Descriptive only: midpoint estimates, non-response, geography, role, and year affect this comparison.


In [ ]:
df.groupby('Country')['Salary_Midpoint'].mean().sort_values(ascending=False).head(10)

In [ ]:
arab_df = df[df['Country'].isin(arab_countries)]
arab_df['Country'].value_counts()

avg_salary_by_country = arab_df.groupby('Country')['Salary_Midpoint'].mean().sort_values(ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(data=avg_salary_by_country.reset_index(), y='Country', x='Salary_Midpoint', palette='coolwarm')
plt.title("Average Salary by Arab Country", fontsize=16, fontweight='bold')
plt.xlabel("Average Salary (USD)")
plt.ylabel("Country")

plt.tight_layout()
plt.show();

In [ ]:
Top_country_salary = df.groupby('Country')['Salary_Midpoint'].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(12,6))
sns.barplot(x=Top_country_salary.index, y=Top_country_salary.values, palette='Spectral')
plt.title('Top 10 Countries by Average Salary')
plt.xlabel('Country')
plt.ylabel('Average Salary')
plt.xticks(rotation=45)
plt.show();

In [ ]:
df_filtered_age.groupby('Age_Group')['Salary_Midpoint'].mean()

In [ ]:
avg_salary_by_age = df_filtered_age.groupby('Age_Group')['Salary_Midpoint'].mean()
avg_salary_by_age = avg_salary_by_age.sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=avg_salary_by_age.index, y=avg_salary_by_age.values, palette='viridis')

plt.title('Average Salary by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Average Salary (USD)')
plt.xticks(rotation=45)

for i, (age_group, salary) in enumerate(avg_salary_by_age.items()):
    plt.text(i, salary + 1000, f'${salary:,.0f}', ha='center', va='bottom')

plt.tight_layout()
plt.show();

In [ ]:
df.groupby('Job_Title')['Salary_Midpoint'].unique()

In [ ]:
df_filtered_job.groupby('Job_Title')['Salary_Midpoint'].mean().sort_values(ascending=False).head()

In [ ]:
top_job_salary = (df_filtered_job.groupby('Job_Title')['Salary_Midpoint']
                  .mean().dropna().sort_values(ascending=False).head(5))
plt.figure(figsize=(12, 6))
sns.barplot(x=top_job_salary.values, y=top_job_salary.index, palette='husl')
plt.title('Top 5 Job Titles by Estimated Salary Midpoint')
plt.xlabel('Estimated salary midpoint (USD)')
plt.ylabel('Job Title')
plt.show()


## Programming_Experience

In [ ]:
df.groupby('Gender')['Programming_Experience'].value_counts()

In [ ]:
df['Programming_Experience'].value_counts()
experience_order = ['No Experience', '0-1 Years', '1-3 Years', '3-5 Years', '5-10 Years', '10+ Years']

plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Gender', hue='Programming_Experience', order=['Female', 'Male', 'Other'], hue_order=experience_order,palette='Set2')
plt.title("Distribution of Programming Experience by Gender")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.legend(title="Programming Experience", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show();


In [ ]:
experience_order = ['No Experience', '0-1 Years', '1-3 Years', '3-5 Years', '5-10 Years', '10+ Years']

plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Education', hue='Programming_Experience', hue_order=experience_order,palette='Set2')
plt.title("Distribution of Programming Experience by Education Level")
plt.xlabel("Education Level")
plt.ylabel("Count")
plt.legend(title="Programming Experience", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.xticks(rotation=45)
plt.show();


## Programming Languages

In [ ]:
lang_counts = df_languages.notna().sum().sort_values(ascending=False)


lang_summary = lang_counts.reset_index()
lang_summary.columns = ["language", "count"]

print(lang_summary)


In [ ]:
language_counts = df_languages.stack().value_counts()

plt.figure(figsize=(12, 6))
sns.barplot(x=language_counts.index, y=language_counts.values, palette='viridis')
plt.title('Usage of Programming Languages')
plt.xlabel('Programming Language')
plt.ylabel('Number of Respondents')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show();


In [ ]:
df_langs_by_year = df[['Year'] + language_cols].copy()

df_melted = df_langs_by_year.melt(id_vars='Year', 
                                  value_vars=language_cols, 
                                  var_name='Language', 
                                  value_name='Used')



df_melted = df_melted.dropna()

lang_year_counts = df_melted.groupby(['Year','Language']).size().unstack(fill_value=0)


lang_year_counts.T.plot(kind='bar', figsize=(12,7))
plt.xlabel("Programming Languages")
plt.ylabel("Count")
plt.title("Programming Languages Usage by Year")
plt.legend(title="Year")
plt.xticks(rotation=45)
plt.show();



## Most_Recommended_Language

In [ ]:
df['Recommended_Language'].value_counts().head(10)

In [ ]:
Most_Recommended_Language = df['Recommended_Language'].value_counts().head(10)
plt.figure(figsize=(12,6))
sns.barplot(x=Most_Recommended_Language.index, y=Most_Recommended_Language.values,palette='coolwarm')
plt.title('Most Recommended Languages')
plt.xlabel('Recommended Language')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show();

In [ ]:
 # حساب تكرار استخدام كل لغة برمجية عبر السنوات
language_year_counts = df.groupby(['Recommended_Language', 'Year']).size().reset_index(name='Count')

# ترتيب اللغات حسب الإجمالي وتحديد أهم 10
top_languages = df['Recommended_Language'].value_counts().head(10).index
language_year_counts_top = language_year_counts[language_year_counts['Recommended_Language'].isin(top_languages)]

# رسم المخطط
plt.figure(figsize=(15, 8))
sns.barplot(data=language_year_counts_top, x='Recommended_Language', y='Count', hue='Year', palette='coolwarm')
plt.title('Top 10 Most Recommended Languages Over Years')
plt.xlabel('Recommended Language')
plt.ylabel('Number of Recommendations')
plt.xticks(rotation=45)
plt.legend(title='Year', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show();

In [ ]:
heatmap_data = pd.crosstab(df['Job_Title'], df['Recommended_Language'])

top_10_jobs = df_filtered_job['Job_Title'].value_counts().head(10).index
heatmap_data = heatmap_data[heatmap_data.index.isin(top_10_jobs)]


top_5_languages = df['Recommended_Language'].value_counts().head(5).index
heatmap_data = heatmap_data[top_5_languages]
plt.figure(figsize=(10, 8))
sns.heatmap(
    heatmap_data,
    annot=True,          
    fmt='d',              
    cmap='Oranges',        
    linewidths=0.5,     
    cbar_kws={'label': 'Number of Recommendations'}
)

plt.title('Heatmap: Recommended Programming Languages by Job Title', fontsize=14, pad=20)
plt.xlabel('Programming Language')
plt.ylabel('Job Title')
plt.tight_layout()
plt.show();

## Education

In [ ]:
df["Education"].value_counts()

In [ ]:
Distrub_by_Edu = df["Education"].value_counts().index
plt.figure(figsize=(10,7))
sns.countplot(data=df, x='Education', order=Distrub_by_Edu, palette='Blues_r')
plt.title("Distribution of Education Levels")
plt.xlabel("Education Level")
plt.ylabel("Number of Respondents")
plt.xticks(rotation=45)
plt.tight_layout()  
plt.show();


In [ ]:
df.groupby('Gender')["Education"].value_counts()

In [ ]:
#1. Create a recurring table (Count)

cross_tab = pd.crosstab(df['Gender'], df['Education'], normalize='index') * 100

#2. Convert to long format for drawing

cross_tab = cross_tab.reset_index().melt(id_vars='Gender', var_name='Education', value_name='Percentage')

#3. Graphing percentages
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=cross_tab, x='Education', y='Percentage', hue='Gender', palette='Set2')

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', fontsize=10,  )

plt.title("Percentage of Education Level by Gender")
plt.xlabel("Education Level")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=45)
plt.legend(title="Gender")
plt.tight_layout()
plt.show();

## Employer_ML_Adoption

In [ ]:
df['Employer_ML_Adoption'].value_counts()

In [ ]:
Adop_of_ML = df["Employer_ML_Adoption"].value_counts().index
plt.figure(figsize=(10, 6))
sns.countplot(data=df, y='Employer_ML_Adoption', palette='Blues_r', order=Adop_of_ML)
plt.title('Employer Machine Learning Adoption Level')
plt.xlabel('Number of Employers')
plt.ylabel('Machine Learning Adoption Level')
plt.tight_layout()
plt.show();

# 6. Insights & Recommendations

# Global Data Science Survey: Verified Portfolio Summary (2017–2021)

## Dataset Scope

This project combines **106,301 survey responses** across five Kaggle Machine Learning & Data Science Survey editions:

| Year | Responses |
|---|---:|
| 2017 | 16,716 |
| 2018 | 23,859 |
| 2019 | 19,717 |
| 2020 | 20,036 |
| 2021 | 25,973 |

Counts represent survey responses, not unique individuals across years. Question wording and availability changed between editions, so comparisons must use harmonized fields and year-aware denominators.

## Verified Participation Findings

- **25–34** is the largest harmonized age group with **39,892 responses (37.5%)**.
- **18–24** follows with **34,821 responses (32.8%)**.
- India recorded **25,192 responses (23.7%)** and the United States **16,885 (15.9%)**.
- Among binary gender responses, **82.0%** were men and **18.0%** women. This describes survey participation, not the global workforce.
- Python was recommended first in **66,892 responses** and was the most frequently selected regular-use language (**65,942 selections**).

## Roles

After preserving unmatched job titles and consolidating only clear synonyms:

- Student: **21,242**
- Data Scientist: **17,128**
- Software Engineer: **12,473**
- Data Analyst: **8,509**
- Research Scientist: **6,968**
- Business Analyst: **4,112**
- Machine Learning Engineer: **3,198**

## Defined Arab-Country Subset

For the seven explicitly selected countries—Egypt, Morocco, Tunisia, Saudi Arabia, United Arab Emirates, Algeria, and Iraq—the dataset contains **2,292 responses**. Egypt leads this subset with **945** responses. Among binary gender responses in this subset, **26.4%** were women. Students represented **21.7%** of the subset.

This is a selected-country comparison, not a complete definition of the Arab world.

## Salary Interpretation

Salary-band midpoints are useful for descriptive exploration but are not proof of equal or unequal pay. Differences can reflect country, role, experience, year, non-response, and survey-design changes. Any causal or policy conclusion requires a controlled analysis.

## Recommendations

1. Use year-specific denominators for trend reporting.
2. Preserve original categories and document every harmonization rule.
3. Present participation patterns as survey evidence, not workforce population estimates.
4. Treat salary comparisons as descriptive until adjusted for role, geography, experience, and year.
5. Continue prioritizing Python and SQL for broad career relevance while tailoring skill development to role.
